In [1]:
import tensorflow as tf
print(tf.__version__)

2.20.0


In [20]:
import pandas as pd
import numpy as np

#libraries for preprocessing 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# libraries for deep learning
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, LeakyReLU
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2

#library for Evaluation/Testing
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

#library for avoiding warnings
import warnings
warnings.filterwarnings("ignore")
import joblib

from sklearn.utils.class_weight import compute_class_weight

In [21]:
dataset = pd.read_csv(r"drug_regulatory_classification_dataset.csv")
dataset.head()

,Dosage_mg,Price_Per_Unit,Production_Cost,Marketing_Spend,Clinical_Trial_Phase,Side_Effect_Severity_Score,Abuse_Potential_Score,Prescription_Rate,Hospital_Distribution_Percentage,Pharmacy_Distribution_Percentage,...,Manufacturing_Region,Requires_Cold_Storage,OTC_Flag,High_Risk_Substance,Insurance_Coverage_Percentage,Export_Percentage,Online_Sales_Percentage,Brand_Reputation_Score,Doctor_Recommendation_Rate,Target_Regulatory_Class
0,250,364.22,246.49,159132.53,3,4.26,2.52,0.74,57.93,42.07,...,South,No,No,No,64.46,41.88,41.15,6.56,0.47,Non-Regulated Drug
1,500,112.86,73.22,260595.45,2,8.12,1.80,0.48,48.00,52.00,...,East,Yes,Yes,No,93.37,10.30,50.05,4.01,0.44,Non-Regulated Drug
2,200,197.24,100.72,106818.45,3,1.58,1.47,0.93,54.48,45.52,...,North,Yes,Yes,No,64.53,34.21,46.42,5.95,0.16,Non-Regulated Drug
3,500,373.55,264.76,231304.59,4,1.72,1.61,0.54,47.20,52.80,...,North,No,Yes,No,95.21,71.04,49.52,9.65,0.76,Non-Regulated Drug
4,500,353.87,277.29,319403.02,2,9.64,0.45,0.50,86.96,13.04,...,North,No,Yes,No,98.05,70.28,44.90,1.79,0.36,Non-Regulated Drug


In [22]:
dataset.shape

(60000, 30)

In [23]:
dataset.isnull().sum()

Dosage_mg                              0
Price_Per_Unit                         0
Production_Cost                        0
Marketing_Spend                        0
Clinical_Trial_Phase                   0
Side_Effect_Severity_Score             0
Abuse_Potential_Score                  0
Prescription_Rate                      0
Hospital_Distribution_Percentage       0
Pharmacy_Distribution_Percentage       0
Annual_Sales_Volume                    0
Regulatory_Risk_Score                  0
Approval_Time_Months                   0
Patent_Duration_Years                  0
R&D_Investment_Million                 0
Competitor_Count                       0
Recall_History_Count                   0
Adverse_Event_Reports                  0
Drug_Form                              0
Therapeutic_Class                      0
Manufacturing_Region                   0
Requires_Cold_Storage                  0
OTC_Flag                               0
High_Risk_Substance                    0
Insurance_Covera

In [24]:
x = dataset.iloc[:, :-1]
y = dataset.iloc[:, -1]

In [25]:
x.head()

,Dosage_mg,Price_Per_Unit,Production_Cost,Marketing_Spend,Clinical_Trial_Phase,Side_Effect_Severity_Score,Abuse_Potential_Score,Prescription_Rate,Hospital_Distribution_Percentage,Pharmacy_Distribution_Percentage,...,Therapeutic_Class,Manufacturing_Region,Requires_Cold_Storage,OTC_Flag,High_Risk_Substance,Insurance_Coverage_Percentage,Export_Percentage,Online_Sales_Percentage,Brand_Reputation_Score,Doctor_Recommendation_Rate
0,250,364.22,246.49,159132.53,3,4.26,2.52,0.74,57.93,42.07,...,Antibiotic,South,No,No,No,64.46,41.88,41.15,6.56,0.47
1,500,112.86,73.22,260595.45,2,8.12,1.80,0.48,48.00,52.00,...,Antidepressant,East,Yes,Yes,No,93.37,10.30,50.05,4.01,0.44
2,200,197.24,100.72,106818.45,3,1.58,1.47,0.93,54.48,45.52,...,Antiviral,North,Yes,Yes,No,64.53,34.21,46.42,5.95,0.16
3,500,373.55,264.76,231304.59,4,1.72,1.61,0.54,47.20,52.80,...,Cardiovascular,North,No,Yes,No,95.21,71.04,49.52,9.65,0.76
4,500,353.87,277.29,319403.02,2,9.64,0.45,0.50,86.96,13.04,...,Antidepressant,North,No,Yes,No,98.05,70.28,44.90,1.79,0.36


In [26]:
y.head()

0    Non-Regulated Drug
1    Non-Regulated Drug
2    Non-Regulated Drug
3    Non-Regulated Drug
4    Non-Regulated Drug
Name: Target_Regulatory_Class, dtype: str

In [27]:
y = y.astype(str).str.strip().str.lower()

In [28]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y = le.fit_transform(y)

num_classes = len(np.unique(y))

In [29]:
numeric_feature = x.select_dtypes(exclude=["object"]).columns
catagorical_feature = x.select_dtypes(include=["object"]).columns

In [30]:
numeric_feature

Index(['Dosage_mg', 'Price_Per_Unit', 'Production_Cost', 'Marketing_Spend',
       'Clinical_Trial_Phase', 'Side_Effect_Severity_Score',
       'Abuse_Potential_Score', 'Prescription_Rate',
       'Hospital_Distribution_Percentage', 'Pharmacy_Distribution_Percentage',
       'Annual_Sales_Volume', 'Regulatory_Risk_Score', 'Approval_Time_Months',
       'Patent_Duration_Years', 'R&D_Investment_Million', 'Competitor_Count',
       'Recall_History_Count', 'Adverse_Event_Reports',
       'Insurance_Coverage_Percentage', 'Export_Percentage',
       'Online_Sales_Percentage', 'Brand_Reputation_Score',
       'Doctor_Recommendation_Rate'],
      dtype='str')

In [31]:
catagorical_feature

Index(['Drug_Form', 'Therapeutic_Class', 'Manufacturing_Region',
       'Requires_Cold_Storage', 'OTC_Flag', 'High_Risk_Substance'],
      dtype='str')

In [32]:
numeric_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

catagorical_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

In [33]:
preprocess_pipeline = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_feature),
    ("catagorical", catagorical_pipeline, catagorical_feature)
])

In [34]:
# split data
X_train, X_test, y_train, y_test = train_test_split(
    x, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [35]:
# preprocess
X_train = preprocess_pipeline.fit_transform(X_train)
X_test = preprocess_pipeline.transform(X_test)

In [36]:
# ══ SAVE ANN PIPELINE — runs after fit_transform, before astype ══
# preprocess_pipeline is now fitted on training DataFrame (29 cols).
# X_test here is still the transformed numpy array — we use x (original) to verify.
import joblib
joblib.dump(preprocess_pipeline, "preprocess_pipeline_ann.pkl")
print("✅ preprocess_pipeline_ann.pkl saved!")
# Verify using original x DataFrame (29 features in, 42 features out after OHE)
test_row = x.iloc[:1]  # original DataFrame, 29 columns
out = preprocess_pipeline.transform(test_row)
print(f"✅ Pipeline verified: 29 cols in → {out.shape[1]} features out")

✅ preprocess_pipeline_ann.pkl saved!
✅ Pipeline verified: 29 cols in → 42 features out


In [37]:
X_train = X_train.astype(np.float32)
X_test = X_test.astype(np.float32)

In [38]:
# convert target to categorical
# y_train = tf.keras.utils.to_categorical(y_train, num_classes)
# y_test_cat = tf.keras.utils.to_categorical(y_test, num_classes)

In [39]:
input_dim = X_train.shape[1]

In [40]:
# improved neural network
model = Sequential([

    Dense(128, activation="relu", input_dim=input_dim),
    BatchNormalization(),
    Dropout(0.3),

    Dense(64, activation="relu"),
    BatchNormalization(),
    Dropout(0.2),

    Dense(32, activation="relu"),

    Dense(num_classes, activation="softmax")
])


In [41]:
# model.compile(
#     optimizer='adam', 
#     loss='binary_crossentropy', 
#     metrics=["accuracy"]
# )

# model.compile(
#     optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
#     loss="categorical_crossentropy",
#     metrics=["accuracy"]
# )

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0008),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


In [42]:
early_stop = EarlyStopping(
    monitor = "val_loss",
    patience = 10, 
    restore_best_weights = True
)

In [43]:
reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.3,
    patience=5
)

In [44]:
history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=120,
    batch_size=32,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

Epoch 1/120
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.7048 - loss: 0.7181 - val_accuracy: 0.7421 - val_loss: 0.6382 - learning_rate: 8.0000e-04
Epoch 2/120
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.7366 - loss: 0.6549 - val_accuracy: 0.7509 - val_loss: 0.6346 - learning_rate: 8.0000e-04
Epoch 3/120
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.7357 - loss: 0.6498 - val_accuracy: 0.7493 - val_loss: 0.6310 - learning_rate: 8.0000e-04
Epoch 4/120
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.7382 - loss: 0.6449 - val_accuracy: 0.7486 - val_loss: 0.6306 - learning_rate: 8.0000e-04
Epoch 5/120
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.7384 - loss: 0.6428 - val_accuracy: 0.7497 - val_loss: 0.6313 - learning_rate: 8.0000e-04
Epoch 6/120
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.7390 - loss: 0.6429 - val_accuracy: 0.7521 - val_loss: 0.6305 - learning_rate: 8.0000e-04
Epoch 7/120
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 2s 

In [45]:
# y_pred = (model.predict(X_test) > 0.5).astype("int32").flatten()

In [46]:
# prediction
y_pred_prob = model.predict(X_test)
y_pred = np.argmax(y_pred_prob, axis=1)

375/375 ━━━━━━━━━━━━━━━━━━━━ 0s 736us/step


In [47]:
print("NaN in y_test:", np.isnan(y_test).sum())

NaN in y_test: 0


In [48]:
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.7428333333333333


In [49]:
print("Classification Report: ", classification_report(y_test, y_pred))

Classification Report:                precision    recall  f1-score   support

           0       0.77      0.92      0.84      8426
           1       0.60      0.40      0.48      2974
           2       0.00      0.00      0.00       600

    accuracy                           0.74     12000
   macro avg       0.46      0.44      0.44     12000
weighted avg       0.69      0.74      0.71     12000



In [50]:
joblib.dump(model, "deep_learning_ANN_model.pkl")

['deep_learning_ANN_model.pkl']

In [51]:
joblib.load("deep_learning_ANN_model.pkl")

<Sequential name=sequential, built=True>